In [ ]:
'''
This script takes the tif download of a Webknossos segmentation and converts
the separated tif images into one single stack and then creates individual
stacks for both the dendrite and the spine mask to be fed into model training
in model training scripts.

Inputs
- Folder of .tifs for each slice from WebKnossos download
- IDs for spine and dendrite mask based on segmentation order in WebKnossos

Outputs
- .tif file of dendrite masks from segmentation for all slices
- .tif file of spine masks from segmentation for all slices
'''

import os
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted

# ------- CONFIGURATION -------
SAMPLE_NAME  = 'example_stack' # Name of the sample and output stacks

# If input data is already a stack look at cell 3
INPUT_FOLDER = 'Inputs/webknossos_download'
OUTPUT_DIR   = 'Output/mask_creation'

# Make sure dendrite and spine IDs are correct, depends on segmentation order in Webknossos
DENDRITE_ID = 1
SPINE_ID    = 2
# ------------------------------

In [ ]:
# Checks every filer in the input folder to see if it's .tif and sorts all of
# the .tif in ascending order based on the number in the filename
sorted_files = natsorted([f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.tif', '.tiff'))])

# Converts each of the images just sorted into a numpy array then stacks them to
# form a single volume
stack = np.stack([tifffile.imread(os.path.join(INPUT_FOLDER, f)) for f in sorted_files])

In [ ]:
# ------------------------------
# If you already have a tif stack just add pathname and use this code instead
# of the above cell
# stack_path = STACK FILE PATH HERE IF YOU HAVE ONE CORRECTLY FORMATTED
# stack = tifffile.imread(stack_path)
# ------------------------------

# Checks the IDs of the stack to make sure there is not too many, meaning the
# stack is likely the raw image, or too few, meaning the stack as no segmentations
# due to improper exporting or other possible reasons
unique_ids = np.unique(stack)
if len(unique_ids) > 100:
    print("WARNING: high pixel value range - might be raw image not segmentaion")
elif any(uid > 0 for uid in unique_ids):
    print(f"\n  Unique IDs found: {unique_ids}")
else:
    print("WARNING: Only background found - check your WebKnossos export settings")

# Makes sure the output directory exists, if it doesn't it will make it
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------
# Can segment more than two things if you add an ID in configuration and add a
# new name in masks
# ------------------------------
# Finds each pixel in the stack with each ID and labels them as such
masks = {
    "dendrite": (stack == DENDRITE_ID),
    "spine": (stack == SPINE_ID),
}

# Saves a .tif of a mask for each ID to the output directory
for name, mask in masks.items():
    out_path = os.path.join(OUTPUT_DIR, f"{SAMPLE_NAME}_{name}.tif")
    tifffile.imwrite(out_path, mask.astype(np.uint8) * 255)
    print(f"Saved {name} mask to {out_path}")

# Takes a random slice of the mask of whatever the first ID is and plots it
# as a quick check to see if it looks correct
# Can also specify the slice by swapping the commented line and putting the
# number of the desired slice.
first_mask_name, first_mask = next(iter(masks.items()))
slice = np.random.randint(0, len(stack))
# slice = 11
plt.figure(figsize=(6, 5))
plt.imshow(first_mask[slice], cmap='gray')
plt.title(f"{SAMPLE_NAME} — {first_mask_name} mask, slice {slice}")
plt.axis('off')
plt.show()